# 1단계 실험: 주제 → 검색 → 요약 (Research Chain)

`src/llm_jacky/research.py` 의 동작을 단계별로 분해해 실험합니다.

**준비**: 프로젝트 루트의 `.env` 에 `OPENAI_API_KEY`, `TAVILY_API_KEY` 가 들어 있어야 합니다.

## 0. 환경 설정

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")

## 1. End-to-end 실행

주제를 바꿔가며 전체 파이프라인을 한 번에 호출해 봅니다.

In [ ]:
from llm_jacky.research import run_research

TOPIC = "외국인을 위한 한국 길거리 음식 추천"

result = run_research(TOPIC, max_results=5)
print("=== 요약 ===")
print(result.summary)
print("\n=== 출처 ===")
for i, s in enumerate(result.sources, 1):
    print(f"[{i}] {s.title}\n    {s.url}")

## 2. 검색 단계만 보기 (Tavily raw 결과)

어떤 자료가 들어오는지 직접 확인합니다. `max_results`, `search_depth` 등을 바꿔 실험해 보세요.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=5)
raw = search.invoke(TOPIC)

print(f"hits: {len(raw)}\n")
for i, r in enumerate(raw, 1):
    print(f"[{i}] {r.get('title')}")
    print(f"    URL: {r.get('url')}")
    snippet = (r.get('content') or '')[:160].replace('\n', ' ')
    print(f"    snippet: {snippet}...\n")

## 3. 프롬프트 확인

LLM 에 실제로 들어가는 프롬프트 텍스트를 점검합니다. 프롬프트 튜닝 시작점.

In [ ]:
from llm_jacky.research import SUMMARY_PROMPT, _format_results

formatted = _format_results(raw)
messages = SUMMARY_PROMPT.format_messages(topic=TOPIC, results=formatted)
for m in messages:
    print(f"--- {m.type.upper()} ---")
    print(m.content[:1200])
    print()

## 4. 요약 단계만 다시 호출 (프롬프트/모델 변경 실험)

검색은 한 번만 하고, 요약 프롬프트나 모델만 바꿔 비교합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

custom_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영어권 독자에게 한국 문화를 소개하는 블로그의 리서처입니다. "
                "검색 결과만 근거로, 인용 가능한 사실 위주로 정리하세요."),
    ("human", "주제: {topic}\n\n검색 결과:\n{results}\n\n"
              "한국어 요약 5문장 + 영어 키워드 5개를 출력하세요."),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = custom_prompt | llm | StrOutputParser()
print(chain.invoke({"topic": TOPIC, "results": formatted}))

## 5. 결과 객체 직렬화 (다음 단계로 넘기기 위해)

2단계(RAG 저장)에서 이 결과를 입력으로 받을 예정이라, 형태를 미리 확인해 둡니다.

In [ ]:
from dataclasses import asdict
import json

payload = {
    "topic": result.topic,
    "summary": result.summary,
    "sources": [asdict(s) for s in result.sources],
}
print(json.dumps(payload, ensure_ascii=False, indent=2)[:1500])